# LLM-as-a-Judge Video Evaluation Demo

This notebook generates a **synthetic CSV** and then evaluates two model outputs using:

- Traditional metrics (classification macro precision/recall/F1 + accuracy)
- ROUGE-L (reference-based text similarity)
- An **LLM-as-a-judge stub** (replaceable with a real judge model/API)
- Paired **bootstrap confidence intervals** and per-scenario reporting

Useful for a 45-minute interview: define schema → compute metrics → slice by scenarios → quantify uncertainty → do failure analysis.

In [ ]:
import os
import re
import hashlib
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

OUT_DIR = "synthetic_eval"
os.makedirs(OUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUT_DIR, "video_llm_eval_synthetic.csv")

SCENARIOS = ["clean", "low_light", "occlusion", "audio_noise", "long_clip", "fast_motion"]
TASK_TYPES = ["classification", "caption", "qa"]

# In Jupyter, IPython.display provides `display`. In other contexts, fall back to printing.
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

print("CSV path:", CSV_PATH)


In [ ]:
# -------------------------------
# Synthetic CSV generator
# -------------------------------
def tokenize(s: str):
    s = (s or "").lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split() if s else []

def lcs_len(a_tokens, b_tokens):
    # LCS length using DP. Strings are short in this synthetic demo.
    n, m = len(a_tokens), len(b_tokens)
    if n == 0 or m == 0:
        return 0
    dp = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(1, n + 1):
        ai = a_tokens[i - 1]
        for j in range(1, m + 1):
            if ai == b_tokens[j - 1]:
                dp[i, j] = dp[i - 1, j - 1] + 1
            else:
                dp[i, j] = max(dp[i - 1, j], dp[i, j - 1])
    return int(dp[n, m])

def rouge_l_f1(pred: str, ref: str):
    # ROUGE-L (F1) using LCS over token sequences.
    p = tokenize(pred)
    r = tokenize(ref)
    if len(p) == 0 and len(r) == 0:
        return 1.0
    if len(p) == 0 or len(r) == 0:
        return 0.0
    lcs = lcs_len(p, r)
    prec = lcs / len(p) if len(p) else 0.0
    rec = lcs / len(r) if len(r) else 0.0
    if prec + rec == 0:
        return 0.0
    return (2 * prec * rec) / (prec + rec)

def difficulty_from_scenario(scenario: str):
    # 0 easy / 1 medium / 2 hard
    base = 0
    if scenario in {"low_light", "occlusion", "audio_noise"}:
        base += 1
    if scenario in {"long_clip", "fast_motion"}:
        base += 1
    return min(2, base)

def gen_reference_text(task_type, scenario, local_rng):
    # Creates a reference output the metrics will compare against.
    people = ["a person", "someone", "a woman", "a man"]
    colors = ["red", "blue", "green", "black", "white", "yellow"]
    actions = ["walking", "sitting", "running", "standing", "waving", "picking up"]
    objects = ["a bag", "a phone", "a cup", "keys", "a bicycle", "a book"]
    places = ["indoors", "outdoors", "on a street", "in a room", "near a door"]

    person = local_rng.choice(people)
    color = local_rng.choice(colors)
    action = local_rng.choice(actions)
    obj = local_rng.choice(objects)
    place = local_rng.choice(places)

    if scenario == "low_light":
        noise_word = "dimly lit"
    elif scenario == "occlusion":
        noise_word = "partially blocked"
    elif scenario == "audio_noise":
        noise_word = "audio is hard to hear"
    elif scenario == "long_clip":
        noise_word = "over a longer duration"
    elif scenario == "fast_motion":
        noise_word = "motion is quick"
    else:
        noise_word = "clear conditions"

    if task_type == "caption":
        return f"{noise_word}, {person} wearing a {color} shirt is {action} and holding {obj} {place}."

    if task_type == "qa":
        # For QA, reference is a short answer string (color/object/place)
        question_type = local_rng.choice(["color", "object", "place"])
        if question_type == "color":
            return f"{color}"
        if question_type == "object":
            return f"{obj}"
        return f"{place}"

    raise ValueError("task_type must be caption or qa for reference text")

def corrupt_text(ref: str, scenario: str, model_quality: float, local_rng):
    # Inject token drops / wrong tokens / extras. Higher quality => fewer corruptions.
    tokens = tokenize(ref)
    diff = difficulty_from_scenario(scenario)
    strength = 0.15 + 0.25 * diff

    drop_p = strength * (1.2 - model_quality)
    swap_p = strength * (0.9 - model_quality)
    extra_p = strength * (0.8 - model_quality)

    new_tokens = list(tokens)
    if new_tokens and drop_p > 0:
        keep = [local_rng.random() > drop_p for _ in new_tokens]
        if any(keep):
            new_tokens = [t for t, k in zip(new_tokens, keep) if k]

    vocab = ["red", "blue", "green", "black", "white", "yellow", "door", "street", "room", "bag", "phone", "cup", "keys"]
    if new_tokens and swap_p > 0:
        for i in range(len(new_tokens)):
            if local_rng.random() < swap_p:
                new_tokens[i] = str(local_rng.choice(vocab))

    if local_rng.random() < extra_p:
        new_tokens += local_rng.choice([["quickly"], ["possibly"], ["seems"], ["definitely"]])

    s = " ".join(new_tokens).replace("  ", " ").strip()
    return s if s else "unknown"

def gen_prediction_text(ref: str, scenario: str, model_quality: float, local_rng):
    # With some probability, produce a closer text; otherwise more corrupted.
    if local_rng.random() < (0.15 + 0.55 * model_quality):
        return corrupt_text(ref, scenario, model_quality=model_quality, local_rng=local_rng)
    return corrupt_text(ref, scenario, model_quality=model_quality * 0.6, local_rng=local_rng)

def gen_classification_example(scenario: str, n_classes: int, model_quality: float, local_rng):
    diff = difficulty_from_scenario(scenario)
    base_acc = 0.55 + 0.18 * (2 - diff)
    acc = base_acc * (0.6 + 0.8 * model_quality)
    acc = float(np.clip(acc, 0.05, 0.99))

    y_true = int(local_rng.integers(0, n_classes))
    if local_rng.random() < acc:
        y_pred = y_true
    else:
        choices = [c for c in range(n_classes) if c != y_true]
        y_pred = int(local_rng.choice(choices))
    return y_true, y_pred

def build_synthetic_csv(n_examples=800, n_classes=7, overwrite=False):
    if (not overwrite) and os.path.exists(CSV_PATH):
        print("CSV already exists; skipping regeneration.")
        return

    rows = []
    # Baseline qualities (A better overall than B)
    modelA_base = 0.88
    modelB_base = 0.78

    for i in range(n_examples):
        video_id = f"vid_{i:04d}"
        task_type = str(rng.choice(TASK_TYPES, p=[0.38, 0.32, 0.30]))
        scenario = str(rng.choice(SCENARIOS, p=[0.25, 0.15, 0.15, 0.15, 0.15, 0.15]))
        diff = difficulty_from_scenario(scenario)

        # Scenario-specific offsets
        qA = modelA_base
        qB = modelB_base
        if scenario in {"low_light", "occlusion"}:
            qA += 0.06
            qB -= 0.03
        if scenario == "audio_noise":
            qB += 0.05
            qA -= 0.02
        if scenario in {"long_clip", "fast_motion"}:
            qA -= 0.03
            qB -= 0.01

        qA = float(np.clip(qA, 0.45, 0.98))
        qB = float(np.clip(qB, 0.45, 0.98))

        row = {
            "example_id": int(i),
            "video_id": video_id,
            "task_type": task_type,
            "scenario": scenario,
            "difficulty_bucket": {0: "easy", 1: "medium", 2: "hard"}[diff],
            # Text fields (caption/qa only)
            "reference_text": np.nan,
            "model_a_text": np.nan,
            "model_b_text": np.nan,
            # Class fields (classification only)
            "ground_truth_class": np.nan,
            "model_a_class": np.nan,
            "model_b_class": np.nan,
        }

        if task_type == "classification":
            y_true, y_pred_a = gen_classification_example(scenario, n_classes, qA, rng)
            _, y_pred_b = gen_classification_example(scenario, n_classes, qB, rng)
            row["ground_truth_class"] = y_true
            row["model_a_class"] = y_pred_a
            row["model_b_class"] = y_pred_b
        else:
            ref = gen_reference_text(task_type=task_type, scenario=scenario, local_rng=rng)
            row["reference_text"] = ref
            row["model_a_text"] = gen_prediction_text(ref, scenario, model_quality=qA, local_rng=rng)
            row["model_b_text"] = gen_prediction_text(ref, scenario, model_quality=qB, local_rng=rng)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    print("Wrote:", CSV_PATH, "rows:", len(df))
    print(df["task_type"].value_counts())

build_synthetic_csv(n_examples=800, n_classes=7, overwrite=True)


In [ ]:
# -------------------------------
# Metrics + Judge stub
# -------------------------------
def compute_macro_precision_recall_f1(y_true, y_pred, labels):
    # Macro averaging over classes.
    precs, recs, f1s = [], [], []
    for c in labels:
        y_true_c = (y_true == c)
        y_pred_c = (y_pred == c)
        tp = int(np.sum(y_true_c & y_pred_c))
        fp = int(np.sum((~y_true_c) & y_pred_c))
        fn = int(np.sum(y_true_c & (~y_pred_c)))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)

    return {
        "precision_macro": float(np.mean(precs)),
        "recall_macro": float(np.mean(recs)),
        "f1_macro": float(np.mean(f1s)),
        "accuracy": float(np.mean(y_true == y_pred)),
    }

def rouge_l_scores(df_sub, pred_col, ref_col="reference_text"):
    # Simple loop keeps this interview-friendly.
    out = []
    for _, r in df_sub.iterrows():
        pred = r[pred_col] if pd.notna(r[pred_col]) else ""
        ref = r[ref_col] if pd.notna(r[ref_col]) else ""
        out.append(rouge_l_f1(str(pred), str(ref)))
    return np.array(out, dtype=float)

def jaccard(a_tokens, b_tokens):
    a = set(a_tokens)
    b = set(b_tokens)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def llm_as_judge_score_stub(reference_text, pred_text, scenario, seed_offset=0):
    # Offline stub that simulates LLM judge variance.
    ref_t = tokenize(str(reference_text))
    pred_t = tokenize(str(pred_text))
    sim = jaccard(ref_t, pred_t)

    diff = difficulty_from_scenario(scenario)
    noise_scale = 0.10 + 0.08 * diff
    local_rng = np.random.default_rng(SEED + seed_offset + 1000 * diff)
    noise = float(local_rng.normal(0, noise_scale))

    sim_clamped = float(np.clip(sim + noise, 0.0, 1.0))
    score_1_5 = 1.0 + 4.0 * sim_clamped
    return float(np.clip(score_1_5, 1.0, 5.0))

def compute_judge_scores(df_sub):
    scores_a, scores_b, prefs = [], [], []
    for _, r in df_sub.iterrows():
        ref = r["reference_text"]
        scn = r["scenario"]
        a = r["model_a_text"]
        b = r["model_b_text"]

        sa = llm_as_judge_score_stub(ref, a, scn, seed_offset=int(r["example_id"]) + 1)
        sb = llm_as_judge_score_stub(ref, b, scn, seed_offset=int(r["example_id"]) + 2)

        scores_a.append(sa)
        scores_b.append(sb)

        if abs(sa - sb) < 0.25:
            coin = np.random.default_rng(SEED + int(r["example_id"]) * 17 + 7).random() < 0.5
            prefs.append(int(coin))
        else:
            prefs.append(int(sa > sb))

    return {
        "judge_score_a": np.array(scores_a, dtype=float),
        "judge_score_b": np.array(scores_b, dtype=float),
        "judge_pref_a": np.array(prefs, dtype=int),
    }

def bootstrap_ci_paired(diff_values, n_boot=2000, alpha=0.05, seed=123):
    diffs = np.asarray(diff_values, dtype=float)
    n = len(diffs)
    if n == 0:
        return {"mean": 0.0, "ci_low": 0.0, "ci_high": 0.0}

    local_rng = np.random.default_rng(seed)
    means = []
    for _ in range(n_boot):
        idx = local_rng.integers(0, n, size=n)
        means.append(float(diffs[idx].mean()))
    means = np.array(means, dtype=float)

    mean_diff = float(diffs.mean())
    ci_low = float(np.quantile(means, alpha / 2))
    ci_high = float(np.quantile(means, 1 - alpha / 2))
    return {"mean": mean_diff, "ci_low": ci_low, "ci_high": ci_high}


In [ ]:
# -------------------------------
# Evaluation runner + scenario slicing
# -------------------------------
df = pd.read_csv(CSV_PATH)

for col in ["ground_truth_class", "model_a_class", "model_b_class"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Loaded rows:", len(df))
print(df["task_type"].value_counts())

def stable_seed_from_str(s: str, mod=10000):
    h = hashlib.md5(s.encode("utf-8")).digest()[:4]
    val = int.from_bytes(h, byteorder="little", signed=False)
    return val % mod

def evaluate_df(df_in):
    results = {}

    # ----- Overall classification -----
    df_cls = df_in[df_in["task_type"] == "classification"].copy()
    if len(df_cls) > 0:
        y_true = df_cls["ground_truth_class"].astype(int).to_numpy()
        y_a = df_cls["model_a_class"].astype(int).to_numpy()
        y_b = df_cls["model_b_class"].astype(int).to_numpy()
        labels = np.unique(y_true)

        m_a = compute_macro_precision_recall_f1(y_true, y_a, labels)
        m_b = compute_macro_precision_recall_f1(y_true, y_b, labels)

        a_corr = (y_a == y_true).astype(float)
        b_corr = (y_b == y_true).astype(float)
        diff_acc = a_corr - b_corr
        acc_ci = bootstrap_ci_paired(diff_acc, n_boot=3000, seed=999)

        results["overall_classification"] = {
            "macro_precision_a": m_a["precision_macro"],
            "macro_precision_b": m_b["precision_macro"],
            "macro_recall_a": m_a["recall_macro"],
            "macro_recall_b": m_b["recall_macro"],
            "macro_f1_a": m_a["f1_macro"],
            "macro_f1_b": m_b["f1_macro"],
            "accuracy_a": m_a["accuracy"],
            "accuracy_b": m_b["accuracy"],
            "paired_accuracy_diff_A_minus_B": acc_ci,
        }

    # ----- Overall text (caption + qa) -----
    df_txt = df_in[df_in["task_type"].isin(["caption", "qa"])].copy()
    if len(df_txt) > 0:
        rouge_a = rouge_l_scores(df_txt, pred_col="model_a_text")
        rouge_b = rouge_l_scores(df_txt, pred_col="model_b_text")
        diff_rouge = rouge_a - rouge_b
        rouge_ci = bootstrap_ci_paired(diff_rouge, n_boot=3000, seed=777)

        judge = compute_judge_scores(df_txt)
        judge_diff = judge["judge_score_a"] - judge["judge_score_b"]
        judge_ci = bootstrap_ci_paired(judge_diff, n_boot=3000, seed=666)

        results["overall_text"] = {
            "rougeL_f1_mean_a": float(np.mean(rouge_a)),
            "rougeL_f1_mean_b": float(np.mean(rouge_b)),
            "paired_rougeL_diff_A_minus_B": rouge_ci,
            "judge_score_1_to_5_mean_a": float(np.mean(judge["judge_score_a"])),
            "judge_score_1_to_5_mean_b": float(np.mean(judge["judge_score_b"])),
            "paired_judge_diff_A_minus_B": judge_ci,
            "judge_preference_rate_for_a": float(np.mean(judge["judge_pref_a"])),
        }

    # ----- Per-scenario slicing -----
    rows = []
    for scn in sorted(df_in["scenario"].dropna().unique()):
        sub = df_in[df_in["scenario"] == scn].copy()

        row = {
            "scenario": scn,
            "n_total": int(len(sub)),
            "n_classification": int(np.sum(sub["task_type"] == "classification")),
            "n_text": int(np.sum(sub["task_type"].isin(["caption", "qa"]))),
        }

        sub_cls = sub[sub["task_type"] == "classification"].copy()
        if len(sub_cls) > 0:
            y_true = sub_cls["ground_truth_class"].astype(int).to_numpy()
            y_a = sub_cls["model_a_class"].astype(int).to_numpy()
            y_b = sub_cls["model_b_class"].astype(int).to_numpy()
            labels = np.unique(y_true)

            m_a = compute_macro_precision_recall_f1(y_true, y_a, labels)
            m_b = compute_macro_precision_recall_f1(y_true, y_b, labels)

            row["classification_accuracy_a"] = m_a["accuracy"]
            row["classification_accuracy_b"] = m_b["accuracy"]
            row["classification_macro_f1_a"] = m_a["f1_macro"]
            row["classification_macro_f1_b"] = m_b["f1_macro"]

            diff_acc = (y_a == y_true).astype(float) - (y_b == y_true).astype(float)
            row["classification_paired_acc_diff_ci"] = bootstrap_ci_paired(
                diff_acc,
                n_boot=1200,
                seed=100 + stable_seed_from_str(scn, mod=1000),
            )

        sub_txt = sub[sub["task_type"].isin(["caption", "qa"])].copy()
        if len(sub_txt) > 0:
            rouge_a = rouge_l_scores(sub_txt, pred_col="model_a_text")
            rouge_b = rouge_l_scores(sub_txt, pred_col="model_b_text")
            row["rougeL_mean_a"] = float(np.mean(rouge_a))
            row["rougeL_mean_b"] = float(np.mean(rouge_b))

            diff_rouge = rouge_a - rouge_b
            row["rougeL_paired_diff_ci"] = bootstrap_ci_paired(
                diff_rouge,
                n_boot=1200,
                seed=200 + stable_seed_from_str(scn, mod=1000),
            )

            judge = compute_judge_scores(sub_txt)
            judge_diff = judge["judge_score_a"] - judge["judge_score_b"]
            row["judge_mean_a"] = float(np.mean(judge["judge_score_a"]))
            row["judge_mean_b"] = float(np.mean(judge["judge_score_b"]))
            row["judge_paired_diff_ci"] = bootstrap_ci_paired(
                judge_diff,
                n_boot=1200,
                seed=300 + stable_seed_from_str(scn, mod=1000),
            )
            row["judge_pref_rate_a"] = float(np.mean(judge["judge_pref_a"]))

        rows.append(row)

    results["scenario_report"] = pd.DataFrame(rows).sort_values("n_total", ascending=False)
    return results

res = evaluate_df(df)

print("\nOverall classification:")
print(res.get("overall_classification", {}))

print("\nOverall text:")
print(res.get("overall_text", {}))

print("\nScenario report (top rows):")
display(res["scenario_report"].head(10))


In [ ]:
# -------------------------------
# Failure analysis (interview-friendly)
# -------------------------------
df_cls = df[df["task_type"] == "classification"].copy()
if len(df_cls) > 0:
    df_cls["a_correct"] = (df_cls["model_a_class"].astype(int) == df_cls["ground_truth_class"].astype(int))
    df_cls["b_correct"] = (df_cls["model_b_class"].astype(int) == df_cls["ground_truth_class"].astype(int))
    disagree = df_cls[df_cls["model_a_class"] != df_cls["model_b_class"]].copy()
    disagree = disagree.sort_values(["a_correct", "b_correct"], ascending=[False, True])
    print("\nClassification disagreements (top 15):")
    display(disagree[[
        "example_id", "video_id", "scenario", "difficulty_bucket",
        "ground_truth_class", "model_a_class", "model_b_class",
        "a_correct", "b_correct"
    ]].head(15))

df_txt = df[df["task_type"].isin(["caption", "qa"])].copy()
if len(df_txt) > 0:
    rouge_a = rouge_l_scores(df_txt, pred_col="model_a_text")
    rouge_b = rouge_l_scores(df_txt, pred_col="model_b_text")
    df_txt["rouge_diff_a_minus_b"] = rouge_a - rouge_b

    judge = compute_judge_scores(df_txt)
    df_txt["judge_diff_a_minus_b"] = judge["judge_score_a"] - judge["judge_score_b"]

    print("\nText examples where A is much better (top 10 by judge diff):")
    display(df_txt.sort_values("judge_diff_a_minus_b", ascending=False)[[
        "example_id", "video_id", "task_type", "scenario", "difficulty_bucket",
        "rouge_diff_a_minus_b", "judge_diff_a_minus_b",
        "model_a_text", "model_b_text", "reference_text"
    ]].head(10))
